# 03 — Cache per-user top-K scores (Phase 1, Step I)

После того как gSASRec обучен (`02_train_gsasrec.ipynb`) и чекпоинт лежит в `artifacts/gsasrec/best.pt`, мы прогоняем его один раз для всех users (train+val или всех экспериментальных) и сохраняем топ-K кандидатов с их скорами.

Этот кэш используется в Phase 2 — групповые агрегаторы читают `s_{u,i}` из этого parquet, а не дёргают gSASRec на каждом батче.

Выход: `artifacts/user_scores_cache/scores.parquet`
Колонки: `uid:int64, item_idx:int64, score:float32, rank:int32`.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

## 1. Восстанавливаем train-данные (тот же canonical pipeline, что и в 02)

In [ ]:
from src.utils.seed import set_seed
from src.data.yambda_loader import (
    load_yambda, filter_listens, filter_min_popularity,
    build_item_id_to_idx, apply_item_remap,
)
from src.data.splits import global_temporal_split, SplitConfig
from src.utils.caching import load_pickle
set_seed(42)

raw = load_yambda('50m', cache_dir=os.environ.get('HF_DATASETS_CACHE'))['interactions']
df = filter_listens(raw)
df = filter_min_popularity(df, min_count=5)

# Используем item_id_to_idx, сохранённый из 02 — гарантирует совпадение индексов.
item_id_to_idx_path = PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'item_id_to_idx.pkl'
item_id_to_idx = load_pickle(item_id_to_idx_path)
df = apply_item_remap(df, item_id_to_idx)
n_items = len(item_id_to_idx)

train, val, test = global_temporal_split(df, SplitConfig())
print(f'train: {len(train):,} events, {train.uid.nunique():,} users, n_items={n_items:,}')

## 2. Загружаем чекпоинт

In [ ]:
from src.scorer.inference import load_checkpoint, cache_user_scores, InferenceConfig
ckpt_path = PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'best.pt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, mcfg = load_checkpoint(ckpt_path, device=device)
max_seq_len = mcfg['max_seq_len']
assert mcfg['n_items'] == n_items, (mcfg['n_items'], n_items)
print(f'model loaded: hidden={mcfg["hidden_dim"]} layers={mcfg["n_layers"]} max_seq_len={max_seq_len}')

## 3. Строим train-последовательности и кэшируем топ-K

In [ ]:
from src.scorer.train import build_user_sequences
sequences = build_user_sequences(train, max_seq_len=max_seq_len)
print(f'sequences: {len(sequences):,} users with >=2 train events')

In [ ]:
K = 200  # для Phase 2 запас, агрегаторы будут резать по своим |C_G|
out_path = PROJECT_ROOT / 'artifacts' / 'user_scores_cache' / 'scores.parquet'

cfg = InferenceConfig(K=K, batch_size=128, exclude_history=True, device=device)
cache_user_scores(model, sequences, max_seq_len, n_items, out_path, cfg)
print('saved:', out_path)

## 4. Sanity-check кэша

In [ ]:
import pandas as pd
scores = pd.read_parquet(out_path)
print('rows:', len(scores))
print('users:', scores.uid.nunique())
print('K per user (should be {}):'.format(K), scores.groupby('uid').size().unique())
print('rank range:', scores['rank'].min(), '..', scores['rank'].max())
print('score range:', float(scores.score.min()), '..', float(scores.score.max()))
scores.head(10)